In [0]:
CREATE OR REPLACE FUNCTION credit_risk_fraud_detection.ml.get_loan_risk_factors(
    loan_id STRING COMMENT 'The loan_account_id to look up current risk-relevant attributes for, e.g. LN_XXXXXXXX'
)
RETURNS TABLE (
    loan_type STRING, persona STRING, cibil_score_at_decision INT,
    consecutive_missed INT, current_dpd INT, dpd_category STRING
)
COMMENT 'Returns the current risk-relevant attributes for a given loan_account_id: loan type, customer persona, CIBIL score at origination, missed payment count, and RBI DPD classification.'
RETURN
  SELECT l.loan_type, c.persona, l.cibil_score_at_decision,
         l.consecutive_missed, l.current_dpd, l.dpd_category
  FROM credit_risk_fraud_detection.silver.dim_loan_account l
  LEFT JOIN credit_risk_fraud_detection.silver.customer_snapshot c
    ON l.customer_id = c.customer_id AND c.dbt_valid_to IS NULL
  WHERE l.loan_account_id = loan_id AND l.is_current;